In [10]:
import pandas as pd
import numpy as np


fact_housing = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\fact_housing_updated.csv')
fact_household = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\fact_household.csv')
dim_lender_terms = pd.read_csv(r'D:\azure_housing_analytics\dataset\gold\support\dim_lender_terms.csv')


<class 'pandas.DataFrame'>
RangeIndex: 17911 entries, 0 to 17910
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17911 non-null  str    
 1   title             17911 non-null  str    
 2   price             17911 non-null  float64
 3   pricePerSqm       15605 non-null  float64
 4   floorArea         11383 non-null  float64
 5   lotArea           15446 non-null  float64
 6   propertyCategory  14065 non-null  str    
 7   locationLevel     17911 non-null  str    
 8   locationFk        17870 non-null  float64
dtypes: float64(5), str(4)
memory usage: 1.2 MB


In [3]:
dim_lender_terms.head()

,channel,ltv_pct,interest_rate,dti_cap_pct,max_loan_amount,max_term_years,source_confidence
0,Pag-IBIG,0.9,0.0575,0.35,10000000.0,30,official
1,Bank (general),0.8,0.0700,0.30,NaN,20,estimated


<class 'pandas.DataFrame'>
RangeIndex: 17911 entries, 0 to 17910
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                17911 non-null  str    
 1   title             17911 non-null  str    
 2   price             17911 non-null  float64
 3   pricePerSqm       15605 non-null  float64
 4   floorArea         11383 non-null  float64
 5   lotArea           15446 non-null  float64
 6   propertyCategory  14065 non-null  str    
 7   locationLevel     17911 non-null  str    
 8   locationFk        17870 non-null  float64
dtypes: float64(5), str(4)
memory usage: 1.2 MB


In [4]:
terms = dim_lender_terms.set_index("channel").to_dict(orient="index")
pagibig = terms["Pag-IBIG"]
bank = terms["Bank (general)"]

def monthly_pmt(principal_col, annual_rate, years):
    r = annual_rate / 12
    n = years * 12
    pmt = principal_col * r * (1 + r) ** n / ((1 + r) ** n - 1)
    return pmt.round(2)


In [9]:
fact_household.info()

<class 'pandas.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   geographyFk           116 non-null    int64  
 1   families              116 non-null    int64  
 2   reliability           116 non-null    str    
 3   monthlyIncome         116 non-null    int64  
 4   monthlyExpenses       116 non-null    int64  
 5   netIncome             116 non-null    int64  
 6   capacityToPayPagibig  116 non-null    float64
 7   capacityToPayBank     116 non-null    float64
dtypes: float64(2), int64(5), str(1)
memory usage: 7.4 KB


In [19]:
city_level = pd.merge(fact_household, fact_housing,left_on='geographyFk', right_on='locationFk' ,how='left')

city_level['geographyFk'] = city_level['locationFk'].where(city_level['locationLevel'] == 'city')

province_level = pd.merge(fact_household, fact_housing,left_on='geographyFk', right_on='locationFk' ,how='left')
province_level['geographyFk'] = province_level['locationFk'].where(province_level['locationLevel'] == 'province')

province_level.head(10)

,geographyFk,families,reliability,monthlyIncome,monthlyExpenses,netIncome,capacityToPayPagibig,capacityToPayBank,id,title,price,pricePerSqm,floorArea,lotArea,propertyCategory,locationLevel,locationFk
0,NaN,510,High,40233,32517,7716,14081.55,12069.9,5e85bcbc-70ae-4747-9287-72d499ae4ebd,Condominium,27705000.0,145235.0,190.76,NaN,Condominium,city,133900000.0
1,NaN,510,High,40233,32517,7716,14081.55,12069.9,81ff202d-3797-4f3d-ae17-bb38a6587d14,With Improvement,7733000.0,27917.0,522.00,277.00,Lot with Improvement,city,133900000.0
2,NaN,510,High,40233,32517,7716,14081.55,12069.9,c0163e2e-7baa-45a3-8d06-a4726dc5449d,With Improvement,2728000.0,46632.0,108.00,58.50,Lot with Improvement,city,133900000.0
3,NaN,510,High,40233,32517,7716,14081.55,12069.9,247cefc8-2e47-421e-9b0d-d26b69aeb626,Townhouse,6493000.0,96766.0,155.18,67.10,Townhouse,city,133900000.0
4,NaN,510,High,40233,32517,7716,14081.55,12069.9,abe8b7e3-4482-4f8c-b34a-b4ef92d97a62,With Improvement-Bundle,20942000.0,74000.0,292.00,283.00,Lot with Improvement,city,133900000.0
5,NaN,510,High,40233,32517,7716,14081.55,12069.9,79298631-0b1e-45ec-82c6-6f132b9f0f1f,With Improvement,247619000.0,343915.0,933.91,720.00,Lot with Improvement,city,133900000.0
6,NaN,510,High,40233,32517,7716,14081.55,12069.9,754f14cc-1fc6-4fbc-a32a-2957b1375614,With Improvement,7548000.0,37930.0,344.00,199.00,Lot with Improvement,city,133900000.0
7,NaN,510,High,40233,32517,7716,14081.55,12069.9,cba2c8ba-0699-4268-a05b-a625ed7593f0,With Improvement,6630000.0,78452.0,157.00,84.51,Lot with Improvement,city,133900000.0
8,NaN,510,High,40233,32517,7716,14081.55,12069.9,77f47006-3501-4c29-b150-81ae2f2dad2d,With Improvement,5640000.0,105283.0,157.00,53.57,Lot with Improvement,city,133900000.0
9,NaN,510,High,40233,32517,7716,14081.55,12069.9,13572d4b-73cd-47ab-8ed6-dd32e48c17a3,With Improvement,6391000.0,77939.0,152.10,82.00,Lot with Improvement,city,133900000.0


In [ ]:
city_level = fact_household.merge(fact_housing, on='geographyFk', how='inner')

df["houseMedianPrice"] = df.groupby("geographyFk")["price"].transform("median")
df["listingCount"] = df.groupby("geographyFk")["price"].transform("count")
# pagibig details
df["pagIbigRequiredLoan"] = df["price"] * pagibig["ltv_pct"]
df["pagIbigAmortization"] = monthly_pmt(
    df["pagIbigRequiredLoan"], pagibig["interest_rate"], pagibig["max_term_years"]
)
df["pagIbigCapacity"] = (df["monthlyIncome"] * pagibig["dti_cap_pct"]).round(2)
df["pagIbigGap"] = (df["pagIbigCapacity"] - df["pagIbigAmortization"]).round(2)
df["pagIbigLoanStatus"] = np.where(df["pagIbigGap"] <= 0, "Not Applicable", "Applicable")


# bank details
df["bankRequiredLoan"] = df["price"] * bank["ltv_pct"]
df["bankAmortization"] = monthly_pmt(
    df["bankRequiredLoan"], bank["interest_rate"], bank["max_term_years"]
)
df["bankCapacity"] = (df["monthlyIncome"] * bank["dti_cap_pct"]).round(2)
df["bankGap"] = (df["bankCapacity"] - df["bankAmortization"]).round(2)
df["bankLoanStatus"] = np.where(df["bankGap"] <= 0, "Not Applicable", "Applicable")


df.head()




,id,sourceSlug,sourceName,title,price,priceFormatted,pricePerSqm,floorArea,lotArea,city,...,pagIbigRequiredLoan,pagIbigAmortization,pagIbigCapacity,pagIbigGap,pagIbigLoanStatus,bankRequiredLoan,bankAmortization,bankCapacity,bankGap,bankLoanStatus
0,b3d16489-85e2-4652-9458-469f0c5d700d,metrobank,Metrobank,Townhouse,5769000.0,₱ 5.8M,52445.0,165.00,110.0,mandaue,...,5192100.0,30299.69,12390.35,-17909.34,Not Applicable,4615200.0,35781.60,10620.3,-25161.30,Not Applicable
1,34d3d3c6-c96c-4ea9-ad7c-8c11d0037370,metrobank,Metrobank,With Improvement,7188000.0,₱ 7.2M,59900.0,296.00,120.0,lapu-lapu,...,6469200.0,37752.50,12490.45,-25262.05,Not Applicable,5750400.0,44582.79,10706.1,-33876.69,Not Applicable
2,5e85bcbc-70ae-4747-9287-72d499ae4ebd,metrobank,Metrobank,Condominium,27705000.0,₱ 27.7M,145235.0,190.76,NaN,manila,...,24934500.0,145510.97,14081.55,-131429.42,Not Applicable,22164000.0,171837.26,12069.9,-159767.36,Not Applicable
3,81ff202d-3797-4f3d-ae17-bb38a6587d14,metrobank,Metrobank,With Improvement,7733000.0,₱ 7.7M,27917.0,522.00,277.0,manila,...,6959700.0,40614.92,14081.55,-26533.37,Not Applicable,6186400.0,47963.09,12069.9,-35893.19,Not Applicable
4,34bbb6f2-e009-4bd9-b3dd-2dba18283cd5,metrobank,Metrobank,Vacant Lot,840000.0,₱ 840K,4800.0,NaN,175.0,tarlac,...,756000.0,4411.81,10728.20,6316.39,Applicable,672000.0,5210.01,9195.6,3985.59,Applicable


In [6]:


to_drop_columns = ['id', 'sourceSlug', 'sourceName', 'title', 'price', 'priceFormatted', 'pricePerSqm', 'floorArea', 'lotArea', 'city', 'province', 'isNew', 'daysListed', 'listingScore', 'firstSeenAt', 'families', 'reliability', 'monthlyIncome', 'monthlyExpenses', 'netIncome']

fact_affordability = df.drop(columns=to_drop_columns)
print(fact_affordability.columns.tolist())

# fact_affordability.to_csv(r'D:\Data_Engineering\Housing_Loan_Clone\azure_housing_analytics\dataset\gold\fact_amortize.csv')
print('table saved')

['geographyFk', 'capacityToPayPagibig', 'capacityToPayBank', 'houseMedianPrice', 'listingCount', 'pagIbigRequiredLoan', 'pagIbigAmortization', 'pagIbigCapacity', 'pagIbigGap', 'pagIbigLoanStatus', 'bankRequiredLoan', 'bankAmortization', 'bankCapacity', 'bankGap', 'bankLoanStatus']
table saved


In [7]:
total_listings = len(dim_housing)
matched_listings = len(df)

print(f"Total listings in dim_housing: {total_listings}")
print(f"Listings matched to fact_household: {matched_listings}")
print(f"Unmatched (no household income data for their geography): {total_listings - matched_listings}")
print(f"Coverage: {matched_listings / total_listings:.1%}")

Total listings in dim_housing: 18224
Listings matched to fact_household: 7800
Unmatched (no household income data for their geography): 10424
Coverage: 42.8%


In [14]:
fact_affordability.listingCount

0        46
1       143
2       781
3       781
4       378
       ... 
7795    548
7796    194
7797    194
7798    194
7799    194
Name: listingCount, Length: 7800, dtype: int64